In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import mlflow
import mlflow.pytorch
import pandas as pd
import glob, os
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import sys
sys.path.append('..')

print("MLflow version:", mlflow.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

d:\nn-xai-agent\xai-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow version: 3.14.0
Device: cpu


In [2]:
class HAMDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = {
            'nv':0,'mel':1,'bkl':2,
            'bcc':3,'akiec':4,'vasc':5,'df':6
        }
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.label_map[row['dx']]

# Reload data
df = pd.read_csv("../data/HAM10000_metadata.csv")
part1 = glob.glob("../data/HAM10000_images_part_1/*.jpg")
part2 = glob.glob("../data/HAM10000_images_part_2/*.jpg")
all_images = part1 + part2
image_paths = {os.path.splitext(os.path.basename(p))[0]: p for p in all_images}
df['image_path'] = df['image_id'].map(image_paths)
df = df.dropna(subset=['image_path'])

# Split
train_df = df.iloc[:int(0.70*len(df))]
val_df   = df.iloc[int(0.70*len(df)):int(0.85*len(df))]

train_transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])
val_transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

train_loader = DataLoader(HAMDataset(train_df, train_transform), batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(HAMDataset(val_df,   val_transform),   batch_size=32, shuffle=False, num_workers=0)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Train: 7010 | Val: 1502


In [3]:
class CNN(nn.Module):
    def __init__(self, num_classes=7):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)
        self.pool  = nn.MaxPool2d(2, 2)
        self.relu  = nn.ReLU()
        self.drop  = nn.Dropout(0.3)
        self.fc1   = nn.Linear(128 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.drop(self.relu(self.fc1(x)))
        return self.fc2(x)

print("CNN defined ✅")

CNN defined ✅


In [12]:
from collections import Counter

def get_class_weights(dataframe):
    label_map = {'nv':0,'mel':1,'bkl':2,'bcc':3,'akiec':4,'vasc':5,'df':6}
    counts = Counter(dataframe['dx'].map(label_map))
    total  = len(dataframe)
    weights = torch.zeros(7)
    for cls, count in counts.items():
        weights[cls] = total / (7 * count)
    return weights.to(device)

def train_with_mlflow(learning_rate, batch_size, epochs, run_name):
    
    with mlflow.start_run(run_name=run_name):
        
        # Log hyperparameters
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("batch_size",    batch_size)
        mlflow.log_param("epochs",        epochs)
        mlflow.log_param("optimizer",     "Adam")
        mlflow.log_param("architecture",  "3-layer CNN")
        mlflow.log_param("dataset",       "HAM10000")
        mlflow.log_param("image_size",    "64x64")
        
        # Setup
        model     = CNN(num_classes=7).to(device)
        weights   = get_class_weights(train_df)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        
        best_val_acc = 0
        
        for epoch in range(epochs):
            # Training
            model.train()
            train_loss, correct, total = 0, 0, 0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                out  = model(images)
                loss = criterion(out, labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                correct    += out.argmax(1).eq(labels).sum().item()
                total      += labels.size(0)
            
            tr_loss = train_loss / len(train_loader)
            tr_acc  = 100. * correct / total
            
            # Validation
            model.eval()
            val_loss, vcorrect, vtotal = 0, 0, 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    out      = model(images)
                    val_loss += criterion(out, labels).item()
                    vcorrect += out.argmax(1).eq(labels).sum().item()
                    vtotal   += labels.size(0)
            
            vl_loss = val_loss / len(val_loader)
            vl_acc  = 100. * vcorrect / vtotal
            
            # Log metrics per epoch
            mlflow.log_metric("train_loss", tr_loss, step=epoch)
            mlflow.log_metric("train_acc",  tr_acc,  step=epoch)
            mlflow.log_metric("val_loss",   vl_loss, step=epoch)
            mlflow.log_metric("val_acc",    vl_acc,  step=epoch)
            
            print(f"Epoch {epoch+1}/{epochs} | "
                  f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.1f}% | "
                  f"Val Loss: {vl_loss:.4f} | Val Acc: {vl_acc:.1f}%")
            
            # Save best model
            if vl_acc > best_val_acc:
                best_val_acc = vl_acc
                torch.save(model.state_dict(), "../model/best_cnn.pth")
                mlflow.log_metric("best_val_acc", best_val_acc)
        
        # Log the final model artifact
        mlflow.pytorch.log_model(
    model,
    "cnn_model",
    serialization_format="pickle"
)
        print(f"\n✅ Run complete. Best val acc: {best_val_acc:.1f}%")
        
    return model

print("Training function ready ✅")

Training function ready ✅


In [5]:
mlflow.set_experiment("ham10000-cnn")
print("Experiment set ✅")

2026/07/07 23:34:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/07 23:34:47 INFO mlflow.store.db.utils: Updating database tables
2026/07/07 23:34:50 INFO mlflow.tracking.fluent: Experiment with name 'ham10000-cnn' does not exist. Creating a new experiment.


Experiment set ✅


In [13]:
print("=== Run 1: Baseline (lr=0.001) ===")
model = train_with_mlflow(
    learning_rate=0.001,
    batch_size=32,
    epochs=3,      # only 3 epochs per run on Day 3 — just to compare runs
    run_name="baseline-lr-0.001"
)

=== Run 1: Baseline (lr=0.001) ===
Epoch 1/3 | Train Loss: 1.4827 | Train Acc: 61.2% | Val Loss: 4.3461 | Val Acc: 2.0%
Epoch 2/3 | Train Loss: 1.0968 | Train Acc: 69.6% | Val Loss: 3.3383 | Val Acc: 1.6%


2026/07/08 00:10:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 3/3 | Train Loss: 1.0113 | Train Acc: 71.2% | Val Loss: 3.2259 | Val Acc: 5.2%


2026/07/08 00:10:00 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/08 00:10:02 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.12.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/08 00:10:16 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.12.1' without the local version label to m


✅ Run complete. Best val acc: 5.2%


In [14]:
print("=== Run 2: Higher LR (lr=0.01) ===")
model = train_with_mlflow(
    learning_rate=0.01,
    batch_size=32,
    epochs=3,
    run_name="high-lr-0.01"
)

=== Run 2: Higher LR (lr=0.01) ===
Epoch 1/3 | Train Loss: 3.2691 | Train Acc: 41.3% | Val Loss: 2.4816 | Val Acc: 7.7%
Epoch 2/3 | Train Loss: 1.6355 | Train Acc: 61.8% | Val Loss: 3.8086 | Val Acc: 13.3%


2026/07/08 00:16:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 3/3 | Train Loss: 1.5824 | Train Acc: 63.2% | Val Loss: 4.4114 | Val Acc: 7.4%


2026/07/08 00:16:35 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/08 00:16:35 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.12.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/08 00:16:44 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.12.1' without the local version label to m


✅ Run complete. Best val acc: 13.3%


In [15]:
print("=== Run 3: Lower LR (lr=0.0001) ===")
model = train_with_mlflow(
    learning_rate=0.0001,
    batch_size=32,
    epochs=3,
    run_name="low-lr-0.0001"
)

=== Run 3: Lower LR (lr=0.0001) ===
Epoch 1/3 | Train Loss: 1.2302 | Train Acc: 68.7% | Val Loss: 3.3913 | Val Acc: 1.9%
Epoch 2/3 | Train Loss: 1.0026 | Train Acc: 73.6% | Val Loss: 3.3655 | Val Acc: 4.9%


2026/07/08 00:22:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 3/3 | Train Loss: 0.9038 | Train Acc: 75.5% | Val Loss: 2.7441 | Val Acc: 6.0%


2026/07/08 00:22:21 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/08 00:22:21 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.12.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/08 00:22:29 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.12.1' without the local version label to m


✅ Run complete. Best val acc: 6.0%


In [16]:
print("Now open a NEW terminal and run:")
print("   mlflow ui")
print("Then open browser at: http://localhost:5000")
print("You will see all 3 runs with their metrics and charts")

Now open a NEW terminal and run:
   mlflow ui
Then open browser at: http://localhost:5000
You will see all 3 runs with their metrics and charts
